In [4]:
"""
Advanced Examples with Larger Datasets

This file contains more advanced examples with realistic data.
Run this to see Word2Vec working on more substantial text data.
"""

from word2vec.word2vec_scratch import Word2VecSkipGram
from word2vec.visualization_utils import (
    get_embedding_stats, analyze_relationships, visualize_embeddings_2d, plot_embeddings_text
)
import numpy as np


In [13]:
# ============================================================================
# EXAMPLE 1: TRAINING ON A REAL DATASET (NEWS-LIKE TEXT)
# ============================================================================
print("=" * 70)
print("EXAMPLE 1: TRAINING ON REALISTIC DATA")
print("=" * 70)

# Create a dataset with some coherent topics
corpus = [
    # Topic: Animals and Nature
    "The lion is a powerful predator that lives in the savanna",
    "Tigers are endangered species found in Asia",
    "Dolphins are intelligent marine mammals",
    "Birds fly through the sky using their wings",
    "Forests contain many different species of plants and animals",
    "Trees provide oxygen and store carbon dioxide",
    "The eagle hunts small animals from high in the sky",
    "Deer are herbivores that live in forests and grasslands",
    "The wolf is a carnivore that hunts in packs",
    "Whales are the largest animals on Earth",

    # Topic: Sports
    "Football players compete on the field",
    "Basketball requires players to shoot the ball into the hoop",
    "Tennis players use rackets to hit the ball",
    "Swimming is a great exercise for fitness",
    "Running is a simple but effective sport",
    "Soccer is the most popular sport in the world",
    "Hockey is played on ice with a stick and puck",
    "Athletes train hard to improve their performance",
    "Teams work together to win matches",
    "Coaches teach players skills and strategy",

    # Topic: Food
    "Pizza is a delicious Italian dish",
    "Bread is made from flour and yeast",
    "Fruits are nutritious and sweet",
    "Vegetables are healthy and contain vitamins",
    "Restaurants serve food to customers",
    "Cooking requires various ingredients and techniques",
    "Rice is a staple food in many Asian countries",
    "Cheese is made from milk",
    "Wine is made from grapes",
    "Desserts are sweet treats",

    # Topic: Technology
    "Computers are powerful machines for processing data",
    "Software runs on computers and performs tasks",
    "Artificial intelligence is transforming technology",
    "Internet connects people around the world",
    "Smartphones are portable computers",
    "Programming is the skill of writing code",
    "Algorithms solve computational problems efficiently",
    "Data science extracts insights from data",
    "Databases store large amounts of information",
    "Networks allow computers to communicate",
]



EXAMPLE 1: TRAINING ON REALISTIC DATA


In [14]:
 # Tokenize
def tokenize(text):
    return text.lower().replace('.', '').replace(',', '').split()

sentences = [tokenize(sent) for sent in corpus]

print(f"Training on {len(sentences)} sentences")
print(f"Sample sentence: {sentences[0]}")


Training on 40 sentences
Sample sentence: ['the', 'lion', 'is', 'a', 'powerful', 'predator', 'that', 'lives', 'in', 'the', 'savanna']


In [15]:
# Train model
model = Word2VecSkipGram(
    embedding_dim=100,
    window_size=3,
    negative_samples=10,
    learning_rate=0.025,
    min_count=1
)

print("\nTraining model (this may take a moment)...")
model.train(sentences, epochs=200, verbose=False)


Training model (this may take a moment)...
Vocabulary size: 175
Negative sampling table size: 999898

Training complete! Total pairs trained: 165674


In [16]:

# Print statistics
get_embedding_stats(model)

# Analyze specific topics
print("\n" + "=" * 70)
print("SEMANTIC ANALYSIS")
print("=" * 70)

topic_words = ["lion", "football", "pizza", "computer"]
analyze_relationships(model, topic_words)

# Try some analogies
print("\n" + "=" * 70)
print("WORD ANALOGIES")
print("=" * 70)

analogy_tests = [
    {
        'positive': ['lion', 'predator'],
        'negative': ['prey'],
        'description': 'lion + predator - prey'
    },
    {
        'positive': ['football', 'team'],
        'negative': ['individual'],
        'description': 'football + team - individual'
    },
]

for test in analogy_tests:
    try:
        print(f"\n{test['description']}:")
        result = model.analogy(test['positive'], test['negative'], topn=3)
        for word, score in result:
            print(f"  - {word}: {score:.4f}")
    except Exception as e:
        print(f"  (Could not compute analogy: {e})")


--- EMBEDDING STATISTICS ---
Vocabulary size: 175
Embedding dimension: 100

Embedding matrix shape: (175, 100)
Mean embedding values: min=-0.8413, max=0.4667, mean=-0.0065
Word vector lengths: min=0.1160, max=2.3625, mean=0.2984
Similarity statistics (cosine): min=0.4360, max=0.9304, mean=0.6778

SEMANTIC ANALYSIS

--- SEMANTIC RELATIONSHIP ANALYSIS ---

LION:
  Vector length: 0.1667
  Most similar words:
    · wolf: 0.9081
    · is: 0.9050
    · a: 0.8678
    · pizza: 0.8553
    · the: 0.8457

FOOTBALL:
  Vector length: 0.1523
  Most similar words:
    · tennis: 0.8439
    · players: 0.8412
    · use: 0.8390
    · requires: 0.8234
    · skills: 0.8168

PIZZA:
  Vector length: 0.1321
  Most similar words:
    · wolf: 0.9056
    · running: 0.9037
    · rice: 0.8934
    · is: 0.8924
    · swimming: 0.8811
'computer' not in vocabulary

WORD ANALOGIES

lion + predator - prey:
  (Could not compute analogy: Word 'prey' not in vocabulary)

football + team - individual:
  (Could not compute a

In [17]:
# ============================================================================
# EXAMPLE 2: DEMONSTRATING CUSTOM TRAINING LOOP
# ============================================================================


In [18]:
"""
Shows how to implement your own training loop for more control.
"""
print("\n" + "=" * 70)
print("EXAMPLE 2: CUSTOM TRAINING LOOP")
print("=" * 70)

sentences = [
    "the sun is bright",
    "bright stars shine",
    "the moon is dark",
    "dark nights are quiet",
    "quiet forests are peaceful",
]



EXAMPLE 2: CUSTOM TRAINING LOOP


In [19]:
def tokenize(text):
    return text.lower().split()

example_sentences = [tokenize(sent) for sent in sentences]

In [20]:
# Create model
model = Word2VecSkipGram(
    embedding_dim=50,
    window_size=2,
    negative_samples=3,
    learning_rate=0.05,
    min_count=1
)

# Build vocabulary
model._build_vocab(example_sentences)
model._init_embeddings()
model._build_negative_sampling_table()

print("\nManual training iteration:")
print("Training on a single sentence multiple times...\n")

sentence = example_sentences[0]
indexed = [model.word2idx[word] for word in sentence if word in model.word2idx]

print(f"Sentence: {sentence}")
print(f"Indices: {indexed}")
print(f"\nTraining 50 iterations on this sentence...\n")

Vocabulary size: 13
Negative sampling table size: 999995

Manual training iteration:
Training on a single sentence multiple times...

Sentence: ['the', 'sun', 'is', 'bright']
Indices: [12, 11, 4, 1]

Training 50 iterations on this sentence...



In [21]:
losses = []
for iteration in range(50):
    total_loss = 0
    pair_count = 0

    for target_pos, target_idx in enumerate(indexed):
        # Define context
        window = 2
        context_start = max(0, target_pos - window)
        context_end = min(len(indexed), target_pos + window + 1)

        for context_pos in range(context_start, context_end):
            if context_pos != target_pos:
                context_idx = indexed[context_pos]
                loss = model._train_pair(target_idx, context_idx)
                total_loss += loss
                pair_count += 1

    avg_loss = total_loss / (pair_count + 1e-8)
    losses.append(avg_loss)

    if (iteration + 1) % 10 == 0:
        print(f"Iteration {iteration+1}: Loss = {avg_loss:.6f}")

print(f"\nLoss over iterations: {losses[0]:.6f} → {losses[-1]:.6f}")
print("(Decreasing loss = model is learning!)")

# Check learned embeddings
print("\nWord similarities after custom training:")
word1, word2 = "sun", "bright"

if word1 in model.word2idx and word2 in model.word2idx:
    vec1 = model.get_vector(word1)
    vec2 = model.get_vector(word2)

    vec1_norm = vec1 / (np.linalg.norm(vec1) + 1e-8)
    vec2_norm = vec2 / (np.linalg.norm(vec2) + 1e-8)

    sim = np.dot(vec1_norm, vec2_norm)
    print(f"Similarity between '{word1}' and '{word2}': {sim:.4f}")


Iteration 10: Loss = 2.772297
Iteration 20: Loss = 2.771887
Iteration 30: Loss = 2.771414
Iteration 40: Loss = 2.771007
Iteration 50: Loss = 2.770988

Loss over iterations: 2.772604 → 2.770988
(Decreasing loss = model is learning!)

Word similarities after custom training:
Similarity between 'sun' and 'bright': 0.7978


In [23]:
# ============================================================================
# EXAMPLE 3: HYPERPARAMETER TUNING
# ============================================================================

"""
Demonstrate how different hyperparameters affect learning.
"""
print("\n" + "=" * 70)
print("EXAMPLE 3: HYPERPARAMETER EFFECTS")
print("=" * 70)

sentences_raw = [
    "deep learning is powerful",
    "neural networks learn representations",
    "machine learning uses deep nets",
    "deep networks have many layers",
    "learning algorithms improve with data",
]

def tokenize(text):
    return text.lower().split()

sentences = [tokenize(sent) for sent in sentences_raw]

print("Testing different embedding dimensions...\n")

test_configs = [
    {'embedding_dim': 10, 'epochs': 50},
    {'embedding_dim': 50, 'epochs': 50},
    {'embedding_dim': 200, 'epochs': 50},
]

for config in test_configs:
    print(f"Config: embedding_dim={config['embedding_dim']}, epochs={config['epochs']}")

    model = Word2VecSkipGram(
        embedding_dim=config['embedding_dim'],
        window_size=2,
        negative_samples=5,
        learning_rate=0.025,
        min_count=1
    )

    model.train(sentences, epochs=config['epochs'], verbose=False)

    # Check similarity
    if 'learning' in model.word2idx and 'deep' in model.word2idx:
        vec1 = model.get_vector('learning')
        vec2 = model.get_vector('deep')

        vec1_norm = vec1 / (np.linalg.norm(vec1) + 1e-8)
        vec2_norm = vec2 / (np.linalg.norm(vec2) + 1e-8)

        sim = np.dot(vec1_norm, vec2_norm)
        print(f"  Similarity(learning, deep): {sim:.4f}")
    print()




EXAMPLE 3: HYPERPARAMETER EFFECTS
Testing different embedding dimensions...

Config: embedding_dim=10, epochs=50
Vocabulary size: 18
Negative sampling table size: 999996

Training complete! Total pairs trained: 2458
  Similarity(learning, deep): 0.8852

Config: embedding_dim=50, epochs=50
Vocabulary size: 18
Negative sampling table size: 999996

Training complete! Total pairs trained: 2450
  Similarity(learning, deep): 0.6064

Config: embedding_dim=200, epochs=50
Vocabulary size: 18
Negative sampling table size: 999996

Training complete! Total pairs trained: 2449
  Similarity(learning, deep): 0.7355

